In [56]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/ecommerce-behavior-data-from-multi-category-store/2019-Nov.csv
/kaggle/input/ecommerce-behavior-data-from-multi-category-store/2019-Oct.csv


**Why are customers not adding to the cart?**

I will start with some clarifying question? 
1. what is the time frame of this behaviour/ is it sudden or abrupt change that happened or it has always been like that ?
2. Demographics
3. Is enagagement same / it has also changed?
4. Is there any external factors like campaign ?
5. Is it seen for certain category of products ?
 

In [57]:
# Installing  dependencies 
import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = "2019-Oct.csv"

# Loading dataset with pandas kwargs
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "mkechinov/ecommerce-behavior-data-from-multi-category-store",
    file_path,
    pandas_kwargs={
        "usecols": ["user_id", "event_type", "event_time",'category_id','category_code','brand','price'],
        "nrows": 500_000   # choosing just the enough rows
    }
)

/tmp/ipykernel_47/282199123.py:8: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


In [58]:
df.head()

,event_time,event_type,category_id,category_code,brand,price,user_id
0,2019-10-01 00:00:00 UTC,view,2103807459595387724,NaN,shiseido,35.79,541312140
1,2019-10-01 00:00:00 UTC,view,2053013552326770905,appliances.environment.water_heater,aqua,33.20,554748717
2,2019-10-01 00:00:01 UTC,view,2053013559792632471,furniture.living_room.sofa,NaN,543.10,519107250
3,2019-10-01 00:00:01 UTC,view,2053013558920217191,computers.notebook,lenovo,251.74,550050854
4,2019-10-01 00:00:04 UTC,view,2053013555631882655,electronics.smartphone,apple,1081.98,535871217


In [59]:
df_counts= df['event_type'].value_counts().reset_index()
df_counts.columns = ["event_type", "count"]
df_details.head()

,event_type,count
0,view,481833
1,purchase,9758
2,cart,8409


looks like people are viewing the product but hesitating to add to cart or purchase it. 
-The purchase count and add to cart need to be investigated. Can user purchase the item without adding it to the cart ? Is there a direct buy now feature in the code ? 


In [60]:
#lets look at the conversion rate
df_counts['conversion_rate']= (df_counts['count'] / df_counts['count'].sum()) *100 
df_counts

,event_type,count,conversion_rate
0,view,481833,96.3666
1,purchase,9758,1.9516
2,cart,8409,1.6818


In the conversion rate we can see that , purchase conversion is only 1.95% and add to cart is only 1.68%

In [61]:
print("Total number of users ")
print(df['user_id'].nunique())

print("Total number of users who viewed")
print(df[df['event_type']=='view']['user_id'].nunique())

print("Total number of users added to cart")
print(df[df['event_type']=='cart']['user_id'].nunique())

print("Total number of users who purchased")
print(df[df['event_type']=='purchase']['user_id'].nunique())


Total number of users 
89124
Total number of users who viewed
89108
Total number of users added to cart
4441
Total number of users who purchased
7362


In [62]:
print("Total number of categories ")
print(df['category_id'].nunique())

print("Total number of categories viewed")
print(df[df['event_type']=='view']['category_id'].nunique())

print("Total number of categories added to cart")
print(df[df['event_type']=='cart']['category_id'].nunique())

print("Total number of categories purchased")
print(df[df['event_type']=='purchase']['category_id'].nunique())


Total number of categories 
540
Total number of categories viewed
540
Total number of categories added to cart
89
Total number of categories purchased
306


To optimize the Add-to-Cart (ATC) functionality and investigate the data discrepancy, you should focus on technical barriers and catalog-wide inconsistencies.

Investigation Strategy: ATC Functionality & Data Integrity
The low ATC ratio suggests a "broken" funnel step between product intent and cart commitment. Use the following structured checks to uncover the root cause:

1. Technical Functionality Audit
Button Visibility: Is the "Add to Cart" button rendered correctly across all browsers and devices (especially mobile)? Check for "hidden" buttons caused by CSS errors or oversized product images.

Interaction Success: Are click events actually firing? Use session recordings to identify "rage clicks" where users click the button but no item is added.

Cart Persistence: Once added, does the item reliably appear in the cart? Test for session timeouts or cookie consent scripts that might be clearing the cart prematurely.

2. Category-Specific Gap Analysis
The 89 vs. 540 Mystery: Investigate the 451 categories with zero ATC activity.

Tracking Issues: Is the ATC tracking tag missing from these specific category templates?

Availability: Are these products perpetually "Out of Stock," or is the button disabled for these specific SKUs?

Content Barriers: Compare the high-performing categories to the low ones—do they lack descriptions, images, or trust signals (like reviews) that provide the "research" value you mentioned?

3. Data Discrepancy Reconciliation
Since Purchases > Carts, the data is likely bypassing the standard funnel:

Direct Checkouts: Check if "Buy It Now" or "Apple/Google Pay" buttons are bypassing the ATC event.

Tracking Lag: Verify if ATC events are being "dropped" by the analytics server while Purchase events (usually on a separate thank-you page) are successfully captured.